# HVAC 2.0 — Stage 2: Fine-tuning main experiment (272 configs / 992 training runs)

Corresponds to `proposal/proposal.md` v6, §7, Stage 2. **The core workload.**

| Part | Grid | Configs | LORO | Training runs |
| --- | --- | --- | --- | --- |
| Small shift | 4 starts × 3 unseen machine IDs × 4 data levels × 4 layer schemes | 192 | — | 192 |
| Large shift | 5 starts × self-recorded fan × 4 data levels × 4 layer schemes | 80 | 10 folds | 800 |
| **Total** | | **272** | | **992** |

Fixed hyperparameters: `lr=1e-4 / epochs=30 / batch=64`, identical for all four layer schemes, no early stopping.
Data levels are nested samples 5 ⊂ 10 ⊂ 20 ⊂ 40, seed=42.

---

### Resuming (Colab can disconnect at any time)

- **The source of truth is `Stage2/logs/stage2_runs.jsonl`**: one line is appended and fsynced right after every **training run** (not every config).
  A large-shift config has 10 folds; saving per run means a disconnect loses at most the run currently in progress.
- **Re-running this notebook = automatic resume**: cell 8 reads the jsonl, skips every completed run, and only fills in the missing ones.
- **A single failure does not stop the loop**: the exception is logged to `stage2_errors.jsonl` and the loop continues; the next re-run retries the failed runs automatically.
- Per-clip scores are written one file per config under `Stage2/logs/clip_scores/` (272 small files), to avoid repeatedly
  appending to one large CSV — Drive's FUSE layer may re-upload the whole file on every close.
- The summary tables `stage2_runs.csv` / `stage2_configs.csv` are derived from the jsonl (cell 10) and can be regenerated at any time.

### Which weights to keep

A single checkpoint is 203 KB (50,760 parameters); even all 992 would be only 200 MB — the proposal's concern was file count, not size.
Compromise: **keep one per config** (the single run for small shift, fold 0 for large shift), 272 files, about 55 MB,
so a representative checkpoint is available for any post-hoc analysis. In addition, **all 992 runs record the per-layer relative weight drift ‖ΔW‖/‖W‖** —
this diagnostic directly answers "where does each layer scheme actually change the model", is more useful than keeping weights, and costs nothing.

---

**How to run**: `Runtime → Run all`. **A T4 GPU runtime is recommended.** This 50k-parameter model is dominated by kernel-launch overhead,
so the GPU advantage over CPU is limited (measured 0.4 ms/step on a local CPU); Colab CPU also works, expected 2–3× slower.

**Embedded config** sha256 = `8430e581ae6b4e14...`, the same file as Stage 0/1.
(This is the hash of the original config file; translating its comments changes the hash, so the sha256 mismatch warning in cell 5 is expected and harmless — splits are read from file, not recomputed.)

## 1 · Environment and dependencies

In [ ]:
import sys, os, re, time, json, glob, hashlib, zipfile, subprocess, traceback, math
from collections import defaultdict, OrderedDict
from datetime import datetime, timedelta

def _ensure(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("pyyaml", "yaml")
_ensure("librosa")
_ensure("tqdm")

import numpy as np
import pandas as pd
import yaml
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import metrics
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python  :", sys.version.split()[0])
print("torch   :", torch.__version__)
print("librosa :", librosa.__version__)
if DEVICE.type == "cuda":
    print("device  : cuda |", torch.cuda.get_device_name(0))
else:
    print("=" * 70)
    print("⚠️  No GPU. It will still run, but expect 2-3x slower than a T4 (about 2-4 hours).")
    print("    Suggestion: Edit → Notebook settings → Hardware accelerator → T4 GPU, then re-run.")
    print("=" * 70)

## 2 · Path constants and run switches ← edit here to adjust

- **Read**: `BASELINE_DIR` (Stage 0 weights and splits), `STAGE1_DIR` (frozen baseline, used to compute gains)
- **Write**: `STAGE2_DIR`
- `LIMIT_RUNS`: maximum number of training runs in this session. Set it to `5` for a small trial run to validate the pipeline;
  results are saved as usual, and after setting it back to `None` the next run picks up where it left off — **nothing is wasted**.
- `KEEP_WEIGHTS`: `"per_config"` (default, one per config) / `"none"` / `"all"`

In [ ]:
# =========================== Edit area ===========================
DRIVE_ROOT    = "/content/drive/MyDrive/HVAC_fan"

MIMII_ZIP     = f"{DRIVE_ROOT}/0_dB_fan.zip"
TARGET_ZIP    = f"{DRIVE_ROOT}/raw_audio.zip"

RESULTS_ROOT  = f"{DRIVE_ROOT}/HVAC2_Results"
BASELINE_DIR  = f"{RESULTS_ROOT}/Baseline"            # Stage 0 (read)  ← note the capital B
STAGE1_DIR    = f"{RESULTS_ROOT}/Stage1"              # Stage 1 (read)
STAGE2_DIR    = f"{RESULTS_ROOT}/Stage2"              # this stage (write)

EXTRACT_ROOT  = "/content/data"                        # local disk: extraction and feature cache

LIMIT_RUNS    = None        # None = run everything; set 5 for a trial run
KEEP_WEIGHTS  = "per_config"   # "per_config" | "none" | "all"
# ===============================================================

BASELINE_MODELS = f"{BASELINE_DIR}/models"
BASELINE_LOGS   = f"{BASELINE_DIR}/logs"
STAGE1_LOGS     = f"{STAGE1_DIR}/logs"
STAGE2_MODELS   = f"{STAGE2_DIR}/models"
STAGE2_LOGS     = f"{STAGE2_DIR}/logs"
STAGE2_SCORES   = f"{STAGE2_LOGS}/clip_scores"

RUNS_JSONL      = f"{STAGE2_LOGS}/stage2_runs.jsonl"      # source of truth for resuming
ERRORS_JSONL    = f"{STAGE2_LOGS}/stage2_errors.jsonl"

MIMII_EXTRACT   = f"{EXTRACT_ROOT}/mimii"
MIMII_ROOT      = f"{MIMII_EXTRACT}/fan"
TARGET_EXTRACT  = f"{EXTRACT_ROOT}/target"
TARGET_ROOT     = f"{TARGET_EXTRACT}/raw_audio"
FEATURE_DIR     = f"{EXTRACT_ROOT}/features"

assert KEEP_WEIGHTS in ("per_config", "none", "all")
for label, p in [("Stage 0 outputs", BASELINE_DIR), ("Stage 1 outputs", STAGE1_DIR), ("Stage 2 outputs", STAGE2_DIR),
                 ("Resume truth", RUNS_JSONL), ("Feature cache", FEATURE_DIR)]:
    print(f"  {label:<16} {p}")
print(f"  LIMIT_RUNS = {LIMIT_RUNS}   KEEP_WEIGHTS = {KEEP_WEIGHTS}")

## 3 · Config (`code/configs/config.yaml` embedded verbatim) + self-check + split rules

The self-check is the same as in Stage 0/1. In addition, several functions from the repository's `code/configs/config.py` are embedded (verbatim):
`rng_for` / `nested_order` / `nested_levels` (nested sampling), `parse_clip` / `loro_split` (LORO),
`enumerate_stage2` / `experiment_id` (experiment enumeration and primary key).

In [ ]:
_CONFIG_YAML_RAW = r"""
# ============================================================================
# HVAC 2.0 — experiment configuration (single source of truth)
# Matches proposal/proposal.md v6. On any conflict, the proposal wins and this file is updated.
# This file holds only "constants and rules"; no training logic.
# ============================================================================

project: HVAC_2.0
proposal_version: v6
seed: 42                      # the one random seed for the whole project (proposal §4.1)

# ---------------------------------------------------------------------------
# Paths. env selects which block is used; keep colab when running on Colab, set local for local debugging.
# ---------------------------------------------------------------------------
paths:
  env: colab                  # colab | local

  colab:
    drive_root: /content/drive/MyDrive
    # Source and target domains are both zips on Drive. Always extract to the Colab local disk /content/data,
    # never back onto Drive: Drive small-file IO is extremely slow, and it would push thousands of wavs to the cloud.
    # The local disk is ephemeral and must be re-extracted after a runtime restart -- extraction should be idempotent (skip if the directory exists).
    extract_root: /content/data

    # Source domain: MIMII 0dB fan
    mimii_zip: /content/drive/MyDrive/HVAC_fan/0_dB_fan.zip
    mimii_extract_to: /content/data/mimii
    mimii_root: /content/data/mimii/fan          # contains id_00/id_02/id_04/id_06

    # Target domain: 180 self-recorded fan clips
    target_zip: /content/drive/MyDrive/HVAC_fan/raw_audio.zip
    target_extract_to: /content/data/target
    target_root: /content/data/target/raw_audio  # contains {condition}/{voltage}/{noise}/
    recording_log: null                          # inventory check only; metadata comes from filenames

    # Outputs: written back to Drive so they survive runtime restarts
    outputs_root: /content/drive/MyDrive/HVAC_fan/HVAC2_Results

  local:
    drive_root: null
    extract_root: null
    # Already extracted locally; no zips needed
    mimii_zip: null
    mimii_extract_to: null
    mimii_root: data/mimii/fan
    target_zip: null
    target_extract_to: null
    target_root: data/raw_audio
    recording_log: data/metadata/recording_log.csv
    outputs_root: outputs

outputs:
  # Stages 2+3 total ~1032 train/eval runs. Weights are discarded after use; only CSVs are saved (proposal §7, stage 2)
  persist_finetuned_weights: false
  pretrained_models_dir: models        # relative to outputs_root; these 5 weights must be kept
  results_dir: logs
  figures_dir: figures

# ---------------------------------------------------------------------------
# Source domain: MIMII fan 0dB (proposal §4.1)
# ---------------------------------------------------------------------------
source_domain:
  name: MIMII_fan_0dB
  machine_ids: [id_00, id_02, id_04, id_06]

  # Official counts, used for a startup self-check: scanned file counts must match exactly, else fail immediately
  official_counts:
    id_00: {normal: 1011, abnormal: 407}
    id_02: {normal: 1016, abnormal: 359}
    id_04: {normal: 1033, abnormal: 348}
    id_06: {normal: 1015, abnormal: 361}

  # Evaluation set: class ratio 1:2, aligned with the target domain's 60:120 (proposal §3)
  eval_set:
    n_normal: 160
    n_abnormal: 320

  # Training pool = all normal clips of that machine ID − the 160 used by the evaluation set
  train_pool_size:
    id_00: 851
    id_02: 856
    id_04: 873
    id_06: 855

  # Merged model: equal contribution from each machine ID
  merged_model:
    # Evaluation set must be a subset of each ID's fixed evaluation set; no independent re-draw (proposal §4.1)
    eval_per_id: {normal: 40, abnormal: 80}
    # Training set: 215 clips drawn from each ID's "training pool", so the total matches the single-ID models
    train_per_id: 215
    train_total: 860

# ---------------------------------------------------------------------------
# Target domain: 180 self-recorded fan clips (proposal §4.2)
# ---------------------------------------------------------------------------
target_domain:
  name: self_recorded_fan
  device: Razer Seiren V3 Mini

  conditions: [normal, blocked, imbalance]
  normal_conditions: [normal]                 # 60 clips, the only part that may enter training
  abnormal_conditions: [blocked, imbalance]   # 120 clips, evaluation only
  voltages: ["4V", "8V", "12V"]
  noise_levels: [quiet, noise]
  n_runs: 10

  n_clips_total: 180
  n_normal_total: 60
  n_abnormal_total: 120

  # Filenames are the single source of truth for metadata; recording_log.csv is only an inventory check
  # (the csv has 206 rows with duplicates and misaligned columns; after dedup, the 180 filenames form a complete 18×10 grid)
  filename_pattern: "{condition}_{voltage}_{noise}_run{run:02d}.wav"
  relpath_pattern: "{condition}/{voltage}/{noise}/{filename}"

# ---------------------------------------------------------------------------
# Feature extraction (proposal §5, identical to baseline_runner.ipynb)
# ---------------------------------------------------------------------------
features:
  sr: 16000
  duration_sec: 10
  n_mels: 64
  frames: 5           # concatenate 5 adjacent frames
  n_fft: 1024
  hop_length: 512
  power: 2.0
  input_dim: 320      # = n_mels * frames
  # Each 10 s clip yields ~309 vectors of dim 320 (313 frames − 5 + 1)

# ---------------------------------------------------------------------------
# Model architecture (proposal §5)
# ---------------------------------------------------------------------------
model:
  arch: MIMII_Baseline_AE
  encoder_dims: [320, 64, 64, 8]
  decoder_dims: [8, 64, 64, 320]
  bottleneck_dim: 8
  activation: relu
  output_activation: none      # official baseline has no activation on the output layer
  # Layer indices inside the torch module, used by the layer schemes to freeze/unfreeze by name
  layer_index:
    encoder_linears: ["encoder.0", "encoder.2", "encoder.4"]   # 320→64, 64→64, 64→8
    decoder_linears: ["decoder.0", "decoder.2", "decoder.4"]   # 8→64, 64→64, 64→320

# ---------------------------------------------------------------------------
# Pretraining (stage 0, proposal §5 / §7 stage 0)
# ---------------------------------------------------------------------------
pretrain:
  optimizer: adam
  lr: 0.001            # torch Adam default, same as baseline_runner.ipynb
  loss: mse
  batch_size: 512
  epochs: 50
  validation_split: 0.1
  shuffle: true
  # Right after training, evaluate once on the model's own 480-clip source eval set and store as the "source baseline" (subtrahend for forgetting)
  record_source_baseline: true

# ---------------------------------------------------------------------------
# Fine-tuning (stage 2) — all four layer schemes share one hyperparameter set, not tuned per scheme (proposal §7, stage 2)
# ---------------------------------------------------------------------------
finetune:
  optimizer: adam
  loss: mse
  lr: 0.0001           # 1/10 of pretraining lr; at the 40-clip full level the movement budget is still < 2.5% of pretraining
  epochs: 30           # derived from a step budget: ~750 updates at the 5-clip level, ~5820 at the 40-clip level
  batch_size: 64       # 512 would give only 4 batches per epoch at the 5-clip level, too few for Adam momentum to warm up
  source_replay: false          # no source-domain replay (proposal §7, stage 2)
  early_stopping: false         # no early stopping; rationale in proposal §7, stage 2
  restart_from_pretrained_each_fold: true   # every LORO fold must restart from the pretrained weights

# ---------------------------------------------------------------------------
# LORO validation protocol (proposal §6)
# ---------------------------------------------------------------------------
loro:
  applies_to: large_shift_only     # no LORO for small shift (proposal §6.4)
  n_folds: 10                      # fold k holds out the whole run == k+1
  fold_key: run                    # determined by runNN in the filename
  clips_per_fold: 18               # one clip from each of the 18 conditions
  normals_per_fold: 6
  abnormals_per_fold: 12
  max_train_normals: 54            # 60 − 6, which is why the data levels stop at 40
  report: [pooled, per_fold]       # compute both views (proposal §6.2)
  per_fold_stats: [mean, std]
  pooled_normalization: none       # no per-fold normalization; listed as a limitation

# ---------------------------------------------------------------------------
# Experiment grid (proposal §7, stage 2)
# ---------------------------------------------------------------------------
grid:
  data_levels: [5, 10, 20, 40]

  # Nested sampling: 5 ⊂ 10 ⊂ 20 ⊂ 40 (proposal §4.3)
  nested_sampling:
    enabled: true
    rule: "Candidates are put in a fixed order by a fixed rule; level n takes the first n"
    # Ordering = sort by filename for a deterministic initial order, then shuffle with a numpy RNG
    canonical_order: filename_sort_then_shuffle
    rng: numpy_default_rng
    # Small shift: order each target ID's training pool once (seed scope includes only the ID)
    small_shift_seed_scope: [seed, target_id]
    # Large shift: available clips differ per fold, so each fold is ordered separately (seed scope includes the fold)
    large_shift_seed_scope: [seed, fold]

  layer_schemes:
    full:
      desc: Full fine-tuning
      trainable: ["encoder.0", "encoder.2", "encoder.4", "decoder.0", "decoder.2", "decoder.4"]
    decoder_only:
      desc: Decoder only
      trainable: ["decoder.0", "decoder.2", "decoder.4"]
    encoder_only:
      desc: Encoder only
      trainable: ["encoder.0", "encoder.2", "encoder.4"]
    bottleneck_only:
      desc: Bottleneck-adjacent layers only (the two layers into and out of the bottleneck, 64→8 and 8→64)
      trainable: ["encoder.4", "decoder.0"]

# ---------------------------------------------------------------------------
# Experiment enumeration per stage (counts are self-checked at startup by config.py; mismatch raises)
# ---------------------------------------------------------------------------
stages:
  stage0_pretrain:
    n_models: 5                  # 4 single-ID + 1 merged
  stage1_frozen_eval:
    small_shift_pairs: 12        # 4 starts × 3 unseen IDs each
    large_shift_models: 5        # all 5 models evaluated on large shift (incl. merged)
    large_shift_folds: 10
    shift_ratio:
      aggregation: median
      denominator: source_eval_normals          # the 160 normals of that ID's eval set
      numerator_small_shift: target_eval_normals # the 160 normals of the target ID's eval set
      numerator_large_shift: all_60_normals      # frozen eval has no leakage, so use all 60 clips
  stage2_finetune:
    small_shift:
      starts: 4
      targets_per_start: 3
      folds: 1
      n_configs: 192
      n_runs: 192
    large_shift:
      starts: 5                  # 4 single-ID + 1 merged
      targets_per_start: 1
      folds: 10
      n_configs: 80
      n_runs: 800
    n_configs_total: 272
    n_runs_total: 992
  stage3_no_pretrain:
    init: random
    layer_scheme: full           # random init has no "learned layers", so no layer-scheme variable
    n_configs: 4
    folds: 10
    n_runs: 40
    source_metrics: null         # no source domain; reported as N/A

total_runs: 1032                 # 992 + 40

# ---------------------------------------------------------------------------
# Evaluation metrics (proposal §3)
# ---------------------------------------------------------------------------
metrics:
  primary: auc                   # floor is always 0.5, independent of class ratio
  secondary: best_f1             # max over a scan of all thresholds
  best_f1_trivial_floor: 0.8     # at a 1:2 class ratio, "predict all abnormal" already gives 0.8
  n_threshold_scan_points: 1000  # matches the linspace in baseline_runner.ipynb
  clip_score: mean_mse_over_frames   # per clip = mean reconstruction MSE over all its feature vectors
  reconstruction_mse_aggregation: median   # threshold shift ratio uses the median
"""
CONFIG_YAML = _CONFIG_YAML_RAW.lstrip("\n")
CONFIG_SHA256 = hashlib.sha256(CONFIG_YAML.encode("utf-8")).hexdigest()
cfg = yaml.safe_load(CONFIG_YAML)


def self_check(cfg):
    src, tgt, grid, stages = cfg["source_domain"], cfg["target_domain"], cfg["grid"], cfg["stages"]
    n_eval_normal, n_eval_abnormal = src["eval_set"]["n_normal"], src["eval_set"]["n_abnormal"]
    for mid in src["machine_ids"]:
        counts = src["official_counts"][mid]
        assert src["train_pool_size"][mid] == counts["normal"] - n_eval_normal
        assert counts["abnormal"] >= n_eval_abnormal
    merged, n_ids = src["merged_model"], len(src["machine_ids"])
    assert merged["eval_per_id"]["normal"] * n_ids == n_eval_normal
    assert merged["eval_per_id"]["abnormal"] * n_ids == n_eval_abnormal
    assert merged["train_per_id"] * n_ids == merged["train_total"]
    pools = list(src["train_pool_size"].values())
    assert min(pools) <= merged["train_total"] <= max(pools)
    src_ratio = n_eval_abnormal / n_eval_normal
    tgt_ratio = tgt["n_abnormal_total"] / tgt["n_normal_total"]
    assert abs(src_ratio - tgt_ratio) < 1e-9
    floor = 2 * src_ratio / (src_ratio + 1) / (1 + src_ratio / (src_ratio + 1))
    assert abs(floor - cfg["metrics"]["best_f1_trivial_floor"]) < 1e-9
    n_cells = len(tgt["conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_cells * tgt["n_runs"] == tgt["n_clips_total"] == 180
    n_normal_cells = len(tgt["normal_conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_normal_cells * tgt["n_runs"] == tgt["n_normal_total"]
    feat = cfg["features"]
    assert feat["n_mels"] * feat["frames"] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][0] == feat["input_dim"]
    loro = cfg["loro"]
    assert loro["n_folds"] == tgt["n_runs"]
    assert loro["clips_per_fold"] == n_cells
    assert loro["normals_per_fold"] == n_normal_cells
    assert loro["max_train_normals"] == tgt["n_normal_total"] - loro["normals_per_fold"]
    levels = grid["data_levels"]
    assert levels == sorted(set(levels)) and max(levels) <= loro["max_train_normals"]
    assert min(levels) >= 1
    known = set(cfg["model"]["layer_index"]["encoder_linears"]) | set(cfg["model"]["layer_index"]["decoder_linears"])
    for name, scheme in grid["layer_schemes"].items():
        assert scheme["trainable"] and not set(scheme["trainable"]) - known, f"layer scheme {name} has invalid layer names"
    assert set(grid["layer_schemes"]["full"]["trainable"]) == known
    n_levels, n_schemes = len(levels), len(grid["layer_schemes"])
    small, large = stages["stage2_finetune"]["small_shift"], stages["stage2_finetune"]["large_shift"]
    assert small["n_configs"] == small["starts"] * small["targets_per_start"] * n_levels * n_schemes
    assert large["n_configs"] == large["starts"] * large["targets_per_start"] * n_levels * n_schemes
    assert small["n_runs"] == small["n_configs"] * small["folds"]
    assert large["n_runs"] == large["n_configs"] * large["folds"]
    assert stages["stage2_finetune"]["n_configs_total"] == small["n_configs"] + large["n_configs"]
    assert stages["stage2_finetune"]["n_runs_total"] == small["n_runs"] + large["n_runs"]
    assert small["starts"] == len(src["machine_ids"])
    assert large["starts"] == stages["stage0_pretrain"]["n_models"]
    ft = cfg["finetune"]
    assert ft["lr"] and ft["epochs"] and ft["batch_size"], "fine-tuning hyperparameters not set"
    assert ft["early_stopping"] is False and ft["source_replay"] is False
    assert ft["restart_from_pretrained_each_fold"] is True


self_check(cfg)

SEED        = cfg["seed"]
MACHINE_IDS = cfg["source_domain"]["machine_ids"]
ALL_MODELS  = MACHINE_IDS + ["merged"]
FEAT        = cfg["features"]
FT          = cfg["finetune"]
LORO        = cfg["loro"]
TGT         = cfg["target_domain"]
GRID        = cfg["grid"]
LEVELS      = GRID["data_levels"]
SCHEMES     = GRID["layer_schemes"]
N_SCAN      = cfg["metrics"]["n_threshold_scan_points"]
F1_FLOOR    = cfg["metrics"]["best_f1_trivial_floor"]
S2          = cfg["stages"]["stage2_finetune"]


# --- The functions below are verbatim copies of those in the repo's code/configs/config.py ---

def rng_for(seed, *parts):
    import zlib
    tag = "|".join(str(p) for p in parts).encode("utf-8")
    return np.random.default_rng([seed, zlib.crc32(tag)])


def nested_order(items, seed, *scope):
    ordered = sorted(items)
    idx = rng_for(seed, *scope).permutation(len(ordered))
    return [ordered[i] for i in idx]


def take_level(ordered, n):
    if n > len(ordered):
        raise ValueError(f"level {n} exceeds the number of available samples {len(ordered)}")
    return list(ordered[:n])


def nested_levels(items, levels, seed, *scope):
    ordered = nested_order(items, seed, *scope)
    out = {n: take_level(ordered, n) for n in levels}
    for smaller, larger in zip(levels, levels[1:]):
        assert out[smaller] == out[larger][:smaller], "nesting property violated"
    return out


CLIP_RE = re.compile(r"^(?P<condition>\w+?)_(?P<voltage>\d+V)_(?P<noise>\w+?)_run(?P<run>\d{2})\.wav$")


def parse_clip(filename):
    m = CLIP_RE.match(os.path.basename(filename))
    if not m:
        raise ValueError(f"filename does not follow the naming convention: {filename}")
    d = m.groupdict()
    d["run"] = int(d["run"])
    return d


def loro_split(cfg, filenames, fold):
    """Fold `fold` (0-based): hold out the whole round run == fold+1 as test. Returns (train_normals, test_clips)."""
    n_folds = cfg["loro"]["n_folds"]
    if not 0 <= fold < n_folds:
        raise ValueError(f"fold must be in [0, {n_folds}), got {fold}")
    held_out_run = fold + 1
    test_clips, train_normals = [], []
    for f in filenames:
        meta = parse_clip(f)
        if meta["run"] == held_out_run:
            test_clips.append(f)
        elif meta["condition"] in cfg["target_domain"]["normal_conditions"]:
            train_normals.append(f)
    loro = cfg["loro"]
    assert len(test_clips) == loro["clips_per_fold"]
    assert len(train_normals) == loro["max_train_normals"]
    return sorted(train_normals), sorted(test_clips)


def target_finetune_sets(cfg, filenames, fold):
    """Large shift: fine-tuning samples for each data level under fold `fold`. Each fold has its own ordering."""
    train_normals, _ = loro_split(cfg, filenames, fold)
    scope = cfg["grid"]["nested_sampling"]["large_shift_seed_scope"]
    parts = [fold if s == "fold" else s for s in scope if s != "seed"]
    return nested_levels(train_normals, cfg["grid"]["data_levels"], cfg["seed"], "large_shift", *parts)


def source_finetune_sets(cfg, train_pool, target_id):
    """Small shift: order the target machine ID's training pool and take each level. train_pool never contains the eval set."""
    return nested_levels(train_pool, cfg["grid"]["data_levels"], cfg["seed"], "small_shift", target_id)


def enumerate_stage2(cfg):
    ids = cfg["source_domain"]["machine_ids"]
    levels = cfg["grid"]["data_levels"]
    schemes = list(cfg["grid"]["layer_schemes"])
    configs = []
    for start in ids:
        for target in [t for t in ids if t != start]:
            for n in levels:
                for scheme in schemes:
                    configs.append(dict(stage="stage2", shift="small", start=start, target=target,
                                        data_level=n, layer_scheme=scheme, n_folds=1, source_eval=start))
    for start in ids + ["merged"]:
        for n in levels:
            for scheme in schemes:
                configs.append(dict(stage="stage2", shift="large", start=start, target="self_recorded_fan",
                                    data_level=n, layer_scheme=scheme, n_folds=cfg["loro"]["n_folds"],
                                    source_eval=start))
    return configs


def experiment_id(c):
    return f"{c['stage']}_{c['shift']}_{c['start']}_to_{c['target']}_n{c['data_level']}_{c['layer_scheme']}"


CONFIGS = enumerate_stage2(cfg)
assert len(CONFIGS) == S2["n_configs_total"] == 272
assert sum(c["n_folds"] for c in CONFIGS) == S2["n_runs_total"] == 992
assert len({experiment_id(c) for c in CONFIGS}) == 272

print(f"Config loaded (proposal {cfg['proposal_version']}), sha256 = {CONFIG_SHA256[:16]}...")
print(f"Self-check passed ✅")
print(f"  grid    : data levels {LEVELS} × layer schemes {list(SCHEMES)}")
print(f"  hparams : lr={FT['lr']} epochs={FT['epochs']} batch={FT['batch_size']} early_stop={FT['early_stopping']}")
print(f"  configs : {len(CONFIGS)} (small shift {S2['small_shift']['n_configs']} + large shift {S2['large_shift']['n_configs']})")
print(f"  runs    : {S2['n_runs_total']} ({S2['small_shift']['n_runs']} + {S2['large_shift']['n_runs']})")

## 4 · Mount Drive, extract, verify directory layout

Same as Stage 1: both zips are extracted to the local disk, idempotently; a mismatch in directory layout or file counts raises an error.

In [ ]:
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for d in (STAGE2_MODELS, STAGE2_LOGS, STAGE2_SCORES):
    os.makedirs(d, exist_ok=True)


def print_tree(base, max_depth=3, max_entries=12):
    base = os.path.abspath(base)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        n_wav = sum(1 for f in files if f.lower().endswith(".wav"))
        print("  " * depth + (os.path.basename(root) or root) + "/" + (f"   [{n_wav} wav]" if n_wav else ""))
        dirs.sort()
        if len(dirs) > max_entries:
            dirs[:] = dirs[:max_entries]


def unzip_idempotent(zip_path, extract_to, ready_fn, what):
    if not os.path.exists(zip_path):
        parent = os.path.dirname(zip_path)
        listing = sorted(os.listdir(parent)) if os.path.isdir(parent) else ["(directory does not exist)"]
        raise FileNotFoundError(f"❌ {what} zip not found: {zip_path}\n   Actual contents of {parent}:\n     "
                                + "\n     ".join(listing) + "\n   → Fix the edit area in cell 2.")
    if ready_fn():
        print(f"✅ {what} already on local disk, skipping extraction")
        return
    os.makedirs(extract_to, exist_ok=True)
    print(f"Extracting {what} ({os.path.getsize(zip_path) / 1024**3:.2f} GB) → {extract_to} ...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        for m in tqdm(zf.infolist(), desc=f"unzip {what}", unit="file"):
            zf.extract(m, extract_to)
    print(f"  Done in {time.time() - t0:.0f} s")


def detect_dir(base, required_subdirs, leaf_subdirs=()):
    for root, dirs, _ in os.walk(base):
        if all(os.path.isdir(os.path.join(root, s)) for s in required_subdirs) and \
           all(os.path.isdir(os.path.join(root, s, l)) for s in required_subdirs for l in leaf_subdirs):
            return root
    return None


# ---- Source domain ----
unzip_idempotent(MIMII_ZIP, MIMII_EXTRACT,
                 lambda: all(os.path.isdir(os.path.join(MIMII_ROOT, m, l)) for m in MACHINE_IDS for l in ("normal", "abnormal")),
                 "source MIMII")
detected = detect_dir(MIMII_EXTRACT, MACHINE_IDS, ("normal", "abnormal"))
if detected is None or os.path.realpath(detected) != os.path.realpath(MIMII_ROOT):
    print(f"❌ Source directory layout mismatch. Expected {MIMII_ROOT}, detected {detected}")
    print_tree(MIMII_EXTRACT)
    raise RuntimeError("Source path problem: set MIMII_ROOT to the detected value and re-run this cell.")
bad = [(m, l, cfg["source_domain"]["official_counts"][m][l], len(glob.glob(os.path.join(MIMII_ROOT, m, l, "*.wav"))))
       for m in MACHINE_IDS for l in ("normal", "abnormal")]
bad = [b for b in bad if b[2] != b[3]]
if bad:
    for m, l, e, g in bad:
        print(f"  {m}/{l}: expected {e}, found {g}")
    raise RuntimeError("Source file counts do not match config: data problem.")
print(f"✅ Source domain verified: {MIMII_ROOT}")

# ---- Target domain ----
unzip_idempotent(TARGET_ZIP, TARGET_EXTRACT,
                 lambda: all(os.path.isdir(os.path.join(TARGET_ROOT, c)) for c in TGT["conditions"]),
                 "target self-recordings")
detected_t = detect_dir(TARGET_EXTRACT, TGT["conditions"])
if detected_t is None or os.path.realpath(detected_t) != os.path.realpath(TARGET_ROOT):
    print(f"❌ Target directory layout mismatch. Expected {TARGET_ROOT}, detected {detected_t}")
    print_tree(TARGET_EXTRACT)
    raise RuntimeError("Target path problem: set TARGET_ROOT to the detected value and re-run this cell.")

target_files = sorted(glob.glob(os.path.join(TARGET_ROOT, "**", "*.wav"), recursive=True),
                      key=os.path.basename)          # sort by filename, independent of directory layout
if len(target_files) != TGT["n_clips_total"]:
    print_tree(TARGET_EXTRACT)
    raise RuntimeError(f"Target domain should have {TGT['n_clips_total']} clips, found {len(target_files)}: data problem.")
grid_chk = defaultdict(list)
for f in target_files:
    d = parse_clip(f)
    grid_chk[(d["condition"], d["voltage"], d["noise"])].append(d["run"])
assert len(grid_chk) == LORO["clips_per_fold"] and all(sorted(v) == list(range(1, TGT["n_runs"] + 1)) for v in grid_chk.values()), \
    "Target 18×10 grid incomplete: data problem, LORO cannot hold out whole rounds"
for k in range(LORO["n_folds"]):
    loro_split(cfg, target_files, k)          # each fold: 54 training normals / 18 test clips; asserted inside the function
print(f"✅ Target domain verified: {TARGET_ROOT} ({len(target_files)} clips, 18×10 grid complete, 10 LORO folds consistent)")

## 5 · Load Stage 0 / Stage 1 outputs

- **Splits**: `Baseline/logs/stage0_splits_seed42.json` — never re-split
- **Source baseline** (subtrahend for forgetting): `Baseline/logs/stage0_source_baseline.csv`
- **Frozen baseline** (subtrahend for gain): `Stage1/logs/stage1_small_shift.csv`, `stage1_large_shift.csv`,
  `stage1_large_shift_folds.csv`. The per-fold large-shift gain is computed against **the same fold** — Stage 1 and this stage
  use identical LORO splits, so each fold tests exactly the same 18 clips.

In [ ]:
def must_exist(p, hint):
    if not os.path.exists(p):
        parent = os.path.dirname(p)
        listing = sorted(os.listdir(parent)) if os.path.isdir(parent) else ["(directory does not exist)"]
        raise FileNotFoundError(f"❌ Not found: {p}\n   Actual contents of {parent}:\n     " + "\n     ".join(listing) + f"\n   → {hint}")
    return p


# ---- Splits ----
with open(must_exist(os.path.join(BASELINE_LOGS, f"stage0_splits_seed{SEED}.json"), "Fix BASELINE_DIR in cell 2 (note the capital B)")) as f:
    blob = json.load(f)
assert blob["seed"] == SEED
if blob.get("config_sha256") != CONFIG_SHA256:
    print(f"⚠️ config sha256 differs from Stage 0 (Stage 0 {str(blob.get('config_sha256'))[:16]}… / this notebook {CONFIG_SHA256[:16]}…). "
          f"Splits are read from file, not recomputed, so correctness is unaffected.")
else:
    print("✅ Split manifest config sha256 matches this notebook")

# Paths relative to MIMII_ROOT are the canonical form of the splits; nested sampling also orders on relative paths
SPLITS_REL = blob["splits"]
splits = {name: {k: [os.path.join(MIMII_ROOT, p) for p in v] for k, v in d.items()} for name, d in SPLITS_REL.items()}
for mid in MACHINE_IDS:
    assert len(SPLITS_REL[mid]["train_pool"]) == cfg["source_domain"]["train_pool_size"][mid]
    assert len(SPLITS_REL[mid]["eval_normal"]) == cfg["source_domain"]["eval_set"]["n_normal"]
    assert len(SPLITS_REL[mid]["eval_abnormal"]) == cfg["source_domain"]["eval_set"]["n_abnormal"]
    assert not set(SPLITS_REL[mid]["train_pool"]) & set(SPLITS_REL[mid]["eval_normal"]), f"{mid} training pool overlaps the eval set"
assert len(SPLITS_REL["merged"]["train_pool"]) == cfg["source_domain"]["merged_model"]["train_total"]

# Small-shift fine-tuning samples per target ID and level (drawn only from the training pool; asserted never to touch the eval set)
SMALL_FT = {}
for tid in MACHINE_IDS:
    SMALL_FT[tid] = source_finetune_sets(cfg, SPLITS_REL[tid]["train_pool"], tid)
    evalset = set(SPLITS_REL[tid]["eval_normal"]) | set(SPLITS_REL[tid]["eval_abnormal"])
    for n, fs in SMALL_FT[tid].items():
        assert len(fs) == n and not set(fs) & evalset, f"{tid} n={n} fine-tuning samples touch the eval set"

# Large-shift fine-tuning samples per fold and level (ordered by filename, independent of directory layout)
TARGET_BY_NAME = {os.path.basename(f): f for f in target_files}
LARGE_FT = {}
for k in range(LORO["n_folds"]):
    LARGE_FT[k] = target_finetune_sets(cfg, list(TARGET_BY_NAME), k)
    _, test_k = loro_split(cfg, list(TARGET_BY_NAME), k)
    for n, fs in LARGE_FT[k].items():
        assert len(fs) == n and not set(fs) & set(test_k), f"fold {k} n={n} fine-tuning samples touch the test round"
        assert all(parse_clip(f)["condition"] in TGT["normal_conditions"] for f in fs)

print(f"✅ Splits and fine-tuning samples ready")
print(f"   Small shift: per target ID 5/10/20/40 ⊂ training pool, asserted not to touch the eval set")
print(f"   Large shift: 10 folds × 5/10/20/40 ⊂ 54 training-side normals, asserted not to touch the test round")
print(f"   e.g. fold0 first 5 clips = {LARGE_FT[0][5]}")

# ---- Source baseline (Stage 0) ----
base_df = pd.read_csv(must_exist(os.path.join(BASELINE_LOGS, "stage0_source_baseline.csv"), "Stage 0 not finished?")).set_index("model")
BASE_SRC_AUC = base_df["auc"].to_dict()
BASE_SRC_F1  = base_df["best_f1"].to_dict()

# ---- Frozen baseline (Stage 1) ----
s1_small = pd.read_csv(must_exist(os.path.join(STAGE1_LOGS, "stage1_small_shift.csv"), "Stage 1 not finished?"))
FROZEN_SMALL = {(r.source_model, r.target_id): {"auc": r.auc, "best_f1": r.best_f1} for r in s1_small.itertuples()}
s1_large = pd.read_csv(must_exist(os.path.join(STAGE1_LOGS, "stage1_large_shift.csv"), "Stage 1 not finished?")).set_index("model")
FROZEN_LARGE = {m: {"pooled_auc": r.pooled_auc, "pooled_best_f1": r.pooled_best_f1,
                    "fold_auc_mean": r.fold_auc_mean, "fold_f1_mean": r.fold_f1_mean}
                for m, r in s1_large.iterrows()}
s1_folds = pd.read_csv(must_exist(os.path.join(STAGE1_LOGS, "stage1_large_shift_folds.csv"), "Stage 1 not finished?"))
FROZEN_FOLD = {(r.model, int(r.fold)): {"auc": r.auc, "best_f1": r.best_f1} for r in s1_folds.itertuples()}
assert len(FROZEN_SMALL) == 12 and len(FROZEN_LARGE) == 5 and len(FROZEN_FOLD) == 50

print(f"\n{'model':<8}{'src AUC (S0)':>16}{'large-shift pooled AUC (S1 frozen)':>36}")
print("-" * 60)
for m in ALL_MODELS:
    print(f"{m:<8}{BASE_SRC_AUC[m]:>16.4f}{FROZEN_LARGE[m]['pooled_auc']:>36.4f}")
print(f"\nSmall-shift frozen AUC (Stage 1): " + ", ".join(f"{s}→{t} {v['auc']:.3f}" for (s, t), v in sorted(FROZEN_SMALL.items())))

## 6 · Feature cache

Same directory and logic as Stage 0/1. If the cache is still there this passes instantly; otherwise it is rebuilt (all MIMII normals + 320 eval abnormals per machine ID,
plus the 180 target clips). **Re-run this cell after a runtime restart.**

In [ ]:
os.makedirs(FEATURE_DIR, exist_ok=True)


def file_to_vector_array(file_name):
    n_mels, frames = FEAT["n_mels"], FEAT["frames"]
    n_fft, hop_length, power = FEAT["n_fft"], FEAT["hop_length"], FEAT["power"]
    y, sr = librosa.load(file_name, sr=None, mono=True)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power)
    log_mel = 20.0 / power * np.log10(mel + sys.float_info.epsilon)
    size = log_mel.shape[1] - frames + 1
    if size < 1:
        return np.empty((0, n_mels * frames), float)
    out = np.zeros((size, n_mels * frames), float)
    for t in range(frames):
        out[:, n_mels * t: n_mels * (t + 1)] = log_mel[:, t: t + size].T
    return out


def build_cache(key, files, base, desc):
    stem = os.path.join(FEATURE_DIR, key)
    npy_path, meta_path = stem + ".npy", stem + ".meta.json"
    rel = [os.path.relpath(f, base) for f in files]
    if os.path.exists(npy_path) and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if set(rel) <= set(meta["files"]):
            return npy_path, meta
    chunks, offsets = [], [0]
    for f in tqdm(files, desc=desc, unit="clip", leave=False):
        v = file_to_vector_array(f).astype(np.float32)
        chunks.append(v)
        offsets.append(offsets[-1] + len(v))
    X = np.concatenate(chunks, axis=0)
    del chunks
    np.save(npy_path, X)
    meta = {"files": rel, "offsets": offsets, "n_clips": len(files), "n_vectors": int(X.shape[0]), "dim": int(X.shape[1])}
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    del X
    return npy_path, meta


t0 = time.time()
CACHE, CACHE_BASE = {}, {}
for mid in MACHINE_IDS:
    need_normal = sorted(set(splits[mid]["train_pool"]) | set(splits[mid]["eval_normal"]))
    for lab, files in (("normal", need_normal), ("abnormal", splits[mid]["eval_abnormal"])):
        key = f"mimii_{mid}_{lab}"
        existed = os.path.exists(os.path.join(FEATURE_DIR, key + ".npy"))
        CACHE[key] = build_cache(key, files, MIMII_ROOT, f"{mid}/{lab}")
        CACHE_BASE[key] = MIMII_ROOT
        print(f"  {key:<24} {len(files):>5} clips  {'cache reused' if existed else 'newly extracted'}")
existed = os.path.exists(os.path.join(FEATURE_DIR, "target_all.npy"))
CACHE["target_all"] = build_cache("target_all", target_files, TARGET_ROOT, "target")
CACHE_BASE["target_all"] = TARGET_ROOT
print(f"  {'target_all':<24} {len(target_files):>5} clips  {'cache reused' if existed else 'newly extracted'}")

per_clip = set()
for key in CACHE:
    off = CACHE[key][1]["offsets"]
    per_clip |= {off[i + 1] - off[i] for i in range(len(off) - 1)}
print(f"\nVectors per clip (set of values): {sorted(per_clip)}  (expected 309)")
print(f"✅ Features ready, this cell took {time.time() - t0:.0f} s")


def load_vectors(key, files):
    """Return (X float32, per-clip slices). files may be absolute paths or paths relative to CACHE_BASE[key]."""
    npy_path, meta = CACHE[key]
    base = CACHE_BASE[key]
    X = np.load(npy_path, mmap_mode="r")
    index = {f: i for i, f in enumerate(meta["files"])}
    off = meta["offsets"]
    parts, slices, cur = [], [], 0
    for f in files:
        rel = os.path.relpath(f, base) if os.path.isabs(f) else f
        i = index[rel]
        seg = np.asarray(X[off[i]:off[i + 1]])
        parts.append(seg)
        slices.append((cur, cur + len(seg)))
        cur += len(seg)
    return np.concatenate(parts, axis=0), slices

## 7 · Model, GPU-resident eval sets, training and evaluation functions

**Eval sets resident in GPU memory**: the 480-clip eval sets of the 4 machine IDs + the merged model's 480 + the 180 target clips,
about 1 GB, loaded once so that the 992 evaluations never re-read from disk.

**Each training run**: restart from pretrained weights → freeze according to the layer scheme → pass only trainable parameters to Adam →
`lr=1e-4 / 30 epochs / batch 64` → **no validation split** (early stopping is excluded, and carving out a validation set would break the nesting contract) →
evaluate on target + source → record per-layer weight drift.

**Vectorized best F1**: all 1000 thresholds are computed at once; results are bit-identical to the point-by-point loop (the Stage 0/1 implementation) —
same thresholds, same `>` comparison, same first-maximum selection. The sklearn point-by-point loop takes about 1 s per call; 992 runs × 2 passes would waste half an hour.

In [ ]:
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 8), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))


LINEAR_LAYERS = cfg["model"]["layer_index"]["encoder_linears"] + cfg["model"]["layer_index"]["decoder_linears"]

# ---- Pretrained weights (kept on CPU, copied for each training run) ----
PRETRAINED = {}
for name in ALL_MODELS:
    p = os.path.join(BASELINE_MODELS, f"baseline_fan_{name}.pth")
    must_exist(p, "Fix BASELINE_DIR in cell 2 (note the capital B)")
    ck = torch.load(p, map_location="cpu", weights_only=False)
    PRETRAINED[name] = {k: v.clone() for k, v in ck["model_state_dict"].items()}
print(f"✅ {len(PRETRAINED)} pretrained checkpoints loaded")


# ---- Eval sets: resident in GPU memory ----
class EvalSet:
    def __init__(self, X, slices, labels, files):
        self.X = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
        self.labels = np.asarray(labels)
        self.files = list(files)
        self.n = len(slices)
        idx = np.zeros(len(X), dtype=np.int64)
        for k, (a, b) in enumerate(slices):
            idx[a:b] = k
        self.clip_idx = torch.from_numpy(idx).to(DEVICE)
        self.counts = torch.bincount(self.clip_idx, minlength=self.n).double()

    def subset(self, mask):
        """Subset by a per-clip boolean mask (for LORO folds); returns a lightweight view."""
        keep = np.where(mask)[0]
        remap = {old: new for new, old in enumerate(keep)}
        rows = torch.isin(self.clip_idx, torch.as_tensor(keep, device=DEVICE))
        sub = object.__new__(EvalSet)
        sub.X = self.X[rows]
        sub.labels = self.labels[keep]
        sub.files = [self.files[i] for i in keep]
        sub.n = len(keep)
        old_idx = self.clip_idx[rows].cpu().numpy()
        sub.clip_idx = torch.from_numpy(np.vectorize(remap.get)(old_idx)).to(DEVICE)
        sub.counts = torch.bincount(sub.clip_idx, minlength=sub.n).double()
        return sub


def build_source_evalset(name):
    s = SPLITS_REL[name]
    if name == "merged":
        parts, slices, cur, files, labels = [], [], 0, [], []
        for mid in MACHINE_IDS:
            for lab, y in (("eval_normal", 0), ("eval_abnormal", 1)):
                fs = [f for f in s[lab] if f.startswith(mid + "/")]
                X, sl = load_vectors(f"mimii_{mid}_{lab.replace('eval_', '')}", fs)
                parts.append(X)
                slices += [(a + cur, b + cur) for a, b in sl]
                cur += len(X)
                files += fs
                labels += [y] * len(fs)
        return EvalSet(np.concatenate(parts), slices, labels, files)
    Xn, sn = load_vectors(f"mimii_{name}_normal", s["eval_normal"])
    Xa, sa = load_vectors(f"mimii_{name}_abnormal", s["eval_abnormal"])
    slices = sn + [(a + len(Xn), b + len(Xn)) for a, b in sa]
    return EvalSet(np.concatenate([Xn, Xa]), slices, [0] * len(sn) + [1] * len(sa), s["eval_normal"] + s["eval_abnormal"])


t0 = time.time()
EVAL = {name: build_source_evalset(name) for name in ALL_MODELS}          # source eval sets (also the small-shift target eval sets)
Xt, st = load_vectors("target_all", target_files)
TARGET_META = [parse_clip(f) for f in target_files]
EVAL["target"] = EvalSet(Xt, st, [0 if d["condition"] in TGT["normal_conditions"] else 1 for d in TARGET_META],
                         [os.path.basename(f) for f in target_files])
TARGET_RUNS = np.array([d["run"] for d in TARGET_META])
FOLD_EVAL = {k: EVAL["target"].subset(TARGET_RUNS == k + 1) for k in range(LORO["n_folds"])}
for k, e in FOLD_EVAL.items():
    assert e.n == LORO["clips_per_fold"] and (e.labels == 0).sum() == LORO["normals_per_fold"]
del Xt
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    print(f"✅ Eval sets resident in GPU memory: {sum(e.X.numel() * 4 for e in EVAL.values()) / 1024**2:.0f} MB, "
          f"GPU memory allocated {torch.cuda.memory_allocated() / 1024**2:.0f} MB, took {time.time() - t0:.0f} s")
else:
    print(f"✅ Eval sets resident in RAM, took {time.time() - t0:.0f} s")


# ---- Training data loading ----
def load_train_vectors(c, fold):
    """Return the fine-tuning vectors (n_vectors, 320) float32 for this config (and fold). Normal samples only."""
    if c["shift"] == "small":
        files = SMALL_FT[c["target"]][c["data_level"]]           # relative paths, from the target ID's training pool
        X, _ = load_vectors(f"mimii_{c['target']}_normal", files)
    else:
        names = LARGE_FT[fold][c["data_level"]]                    # filenames, from the 54 training-side normals
        X, _ = load_vectors("target_all", [TARGET_BY_NAME[n] for n in names])
    return X, len(files) if c["shift"] == "small" else len(names)


# ---- Layer-scheme freezing ----
def apply_layer_scheme(model, scheme_name):
    prefixes = tuple(p + "." for p in SCHEMES[scheme_name]["trainable"])
    trainable = []
    for name, p in model.named_parameters():
        on = name.startswith(prefixes)
        p.requires_grad_(on)
        if on:
            trainable.append(p)
    assert trainable, f"{scheme_name} has no trainable parameters"
    return trainable


# ---- Training ----
def train_one(c, fold, seed):
    """Fine-tune starting fresh from pretrained weights; return (model, info)."""
    torch.manual_seed(seed)
    model = MIMII_Baseline_AE(FEAT["input_dim"]).to(DEVICE)
    model.load_state_dict(PRETRAINED[c["start"]])                  # always restart from pretrained weights
    w_before = {ln: model.get_submodule(ln).weight.detach().clone() for ln in LINEAR_LAYERS}

    params = apply_layer_scheme(model, c["layer_scheme"])
    optimizer = optim.Adam(params, lr=FT["lr"])
    criterion = nn.MSELoss()

    X_np, n_clips = load_train_vectors(c, fold)
    X = torch.from_numpy(np.ascontiguousarray(X_np)).to(DEVICE)
    n, bs, epochs = len(X), FT["batch_size"], FT["epochs"]
    n_batches = (n + bs - 1) // bs
    losses = []
    model.train()
    t0 = time.time()
    for epoch in range(epochs):
        idx = torch.randperm(n, device=DEVICE)
        total = 0.0
        for b in range(n_batches):
            xb = X[idx[b * bs:(b + 1) * bs]]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), xb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        losses.append(total / n_batches)
    train_sec = time.time() - t0
    model.eval()

    drift = {}
    trainable_prefixes = tuple(SCHEMES[c["layer_scheme"]]["trainable"])
    for ln in LINEAR_LAYERS:
        w = model.get_submodule(ln).weight.detach()
        drift[ln] = float((w - w_before[ln]).norm() / w_before[ln].norm())
        if ln not in trainable_prefixes:
            assert drift[ln] == 0.0, f"{ln} should be frozen under {c['layer_scheme']} but drifted by {drift[ln]:.3e}"
    del X, w_before
    return model, {"n_train_clips": n_clips, "n_train_vectors": n, "steps": n_batches * epochs,
                   "n_trainable_params": int(sum(p.numel() for p in params)),
                   "loss_first": losses[0], "loss_last": losses[-1], "loss_curve": [round(v, 6) for v in losses],
                   "train_sec": round(train_sec, 2), "drift": drift}


# ---- Evaluation ----
@torch.no_grad()
def score_evalset(model, ev, chunk=65536):
    """Per-clip anomaly score = mean reconstruction MSE over all frames of the clip. Full-batch forward, aggregated per clip."""
    mse = torch.empty(len(ev.X), device=DEVICE, dtype=torch.float64)
    for a in range(0, len(ev.X), chunk):
        xb = ev.X[a:a + chunk]
        mse[a:a + chunk] = torch.mean((xb - model(xb)) ** 2, dim=1).double()
    s = torch.zeros(ev.n, device=DEVICE, dtype=torch.float64).index_add_(0, ev.clip_idx, mse) / ev.counts
    return s.cpu().numpy()


def best_f1_scan(y_true, y_scores, n_points=None):
    """Bit-identical to the Stage 0/1 point-by-point loop: linspace thresholds, `>` means anomalous, first maximum taken."""
    n_points = N_SCAN if n_points is None else n_points
    y = np.asarray(y_true).astype(bool)
    s = np.asarray(y_scores, dtype=np.float64)
    ts = np.linspace(s.min(), s.max(), n_points)
    pred = s[None, :] > ts[:, None]
    tp = (pred & y).sum(1); fp = (pred & ~y).sum(1); fn = (~pred & y).sum(1); tn = (~pred & ~y).sum(1)
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    i = int(np.argmax(f1))
    if f1[i] <= 0:
        return {"f1": 0.0, "threshold": float("nan"), "precision": 0.0, "recall": 0.0, "accuracy": float(tn[0] + tp[0]) / len(s)}
    return {"f1": float(f1[i]), "threshold": float(ts[i]),
            "precision": float(tp[i] / (tp[i] + fp[i])) if tp[i] + fp[i] else 0.0,
            "recall": float(tp[i] / (tp[i] + fn[i])) if tp[i] + fn[i] else 0.0,
            "accuracy": float((tp[i] + tn[i]) / len(s))}


def evaluate_on(model, ev):
    scores = score_evalset(model, ev)
    return {"auc": float(metrics.roc_auc_score(ev.labels, scores)), **best_f1_scan(ev.labels, scores)}, scores


print(f"✅ Train/eval functions ready | trainable params: " + ", ".join(
    f"{k}={sum(p.numel() for n, p in MIMII_Baseline_AE(320).named_parameters() if n.startswith(tuple(x + '.' for x in v['trainable'])))}"
    for k, v in SCHEMES.items()))

## 8 · Run plan and resume detection

Expand the 272 configs into 992 `run_id = {experiment_id}__fold{k}`, read `stage2_runs.jsonl`
to find the completed ones, and queue the rest in order. Small shift (192 runs, fast) goes first, large shift (800 runs) after.

**The order is fixed and matches `enumerate_stage2()` in the repository's `code/configs/config.py`**, so every re-run produces the same queue.

In [ ]:
def run_id_of(c, fold):
    return f"{experiment_id(c)}__fold{fold}"


PLAN = []                                   # 992 (config, fold) pairs
for c in CONFIGS:
    for k in range(c["n_folds"]):
        PLAN.append((c, k))
assert len(PLAN) == S2["n_runs_total"] == 992
assert len({run_id_of(c, k) for c, k in PLAN}) == 992


def load_done():
    """Read the jsonl and return {run_id: record}. Tolerates a truncated last line (may be half-written on disconnect)."""
    done = {}
    if os.path.exists(RUNS_JSONL):
        with open(RUNS_JSONL) as f:
            for ln, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    done[r["run_id"]] = r
                except json.JSONDecodeError:
                    print(f"  ⚠️ jsonl line {ln} is corrupt (most likely half-written on disconnect), skipped; that run will be redone")
    return done


DONE = load_done()
TODO = [(c, k) for c, k in PLAN if run_id_of(c, k) not in DONE]
n_err_prev = sum(1 for _ in open(ERRORS_JSONL)) if os.path.exists(ERRORS_JSONL) else 0

print(f"Planned {len(PLAN)} runs | done {len(DONE)} | to run {len(TODO)}"
      + (f" | session cap LIMIT_RUNS={LIMIT_RUNS}" if LIMIT_RUNS else "")
      + (f" | {n_err_prev} past failure records (retried automatically from the queue)" if n_err_prev else ""))
if TODO:
    c0, k0 = TODO[0]
    print(f"Next: {run_id_of(c0, k0)}")
    n_small = sum(1 for c, _ in TODO if c["shift"] == "small")
    print(f"Queue: {n_small} small-shift + {len(TODO) - n_small} large-shift runs")
else:
    print("🎉 All done; just run cell 10 for the summary")

## 9 · Main loop (992 training runs)

For each training run, **per-clip scores are written before the jsonl line** — any run in the jsonl is guaranteed to have its score file;
conversely, if a score file has an extra run (disconnect between the two writes), the re-run first removes that run's old lines and then appends, so nothing is duplicated.

The progress bar advances per run; **one line is printed per completed config** (for large shift, one line with aggregate numbers after all 10 folds);
an ETA line is printed every 10 configs.

Single failure: logged to `stage2_errors.jsonl`, then the loop moves on; re-running this notebook retries it automatically.

In [ ]:
def append_jsonl(path, obj):
    with open(path, "a") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def write_clip_scores(eid, run_id, fold, rows):
    """One file per config; first drop old lines with the same run_id (may be left over after a resume), then append."""
    p = os.path.join(STAGE2_SCORES, f"{eid}.csv")
    header = "run_id,fold,domain,file,label,score\n"
    if os.path.exists(p):
        with open(p) as f:
            old = [ln for ln in f.read().splitlines()[1:] if ln and not ln.startswith(run_id + ",")]
    else:
        old = []
    with open(p, "w") as f:
        f.write(header)
        for ln in old:
            f.write(ln + "\n")
        for dom, fn, y, sc in rows:
            f.write(f"{run_id},{fold},{dom},{fn},{int(y)},{sc:.10e}\n")
        f.flush()
        os.fsync(f.fileno())


def do_run(c, fold):
    """Train + evaluate + persist one run. Return the record dict."""
    eid, rid = experiment_id(c), run_id_of(c, fold)
    seed = int(rng_for(SEED, "stage2", eid, fold).integers(0, 2**31 - 1))
    t0 = time.time()

    model, info = train_one(c, fold, seed)

    # target domain
    if c["shift"] == "small":
        tgt_ev, frozen_t = EVAL[c["target"]], FROZEN_SMALL[(c["start"], c["target"])]
    else:
        tgt_ev, frozen_t = FOLD_EVAL[fold], FROZEN_FOLD[(c["start"], fold)]
    tgt, tgt_scores = evaluate_on(model, tgt_ev)

    # source domain (re-evaluated on its own eval set after fine-tuning)
    src_ev = EVAL[c["source_eval"]]
    src, src_scores = evaluate_on(model, src_ev)

    rows = [("target", fn, y, sc) for fn, y, sc in zip(tgt_ev.files, tgt_ev.labels, tgt_scores)] + \
           [("source", fn, y, sc) for fn, y, sc in zip(src_ev.files, src_ev.labels, src_scores)]
    write_clip_scores(eid, rid, fold, rows)

    keep = KEEP_WEIGHTS == "all" or (KEEP_WEIGHTS == "per_config" and fold == 0)
    if keep:
        torch.save({"model_state_dict": model.state_dict(), "input_dim": FEAT["input_dim"], "config": c,
                    "fold": fold, "seed": seed, "config_sha256": CONFIG_SHA256},
                   os.path.join(STAGE2_MODELS, f"{eid}__fold{fold}.pth"))

    rec = {
        "run_id": rid, "experiment_id": eid, "fold": fold, "shift": c["shift"], "start": c["start"],
        "target": c["target"], "data_level": c["data_level"], "layer_scheme": c["layer_scheme"], "seed": seed,
        **{k: v for k, v in info.items() if k != "drift"},
        "target_auc": tgt["auc"], "target_best_f1": tgt["f1"], "target_threshold": tgt["threshold"],
        "target_precision": tgt["precision"], "target_recall": tgt["recall"],
        "frozen_target_auc": frozen_t["auc"], "frozen_target_best_f1": frozen_t["best_f1"],
        "target_auc_gain": tgt["auc"] - frozen_t["auc"], "target_f1_gain": tgt["f1"] - frozen_t["best_f1"],
        "source_auc": src["auc"], "source_best_f1": src["f1"],
        "base_source_auc": BASE_SRC_AUC[c["source_eval"]], "base_source_best_f1": BASE_SRC_F1[c["source_eval"]],
        "source_auc_forget": BASE_SRC_AUC[c["source_eval"]] - src["auc"],
        "source_f1_forget": BASE_SRC_F1[c["source_eval"]] - src["f1"],
        **{f"drift_{ln}": v for ln, v in info["drift"].items()},
        "drift_encoder_mean": float(np.mean([info["drift"][ln] for ln in cfg["model"]["layer_index"]["encoder_linears"]])),
        "drift_decoder_mean": float(np.mean([info["drift"][ln] for ln in cfg["model"]["layer_index"]["decoder_linears"]])),
        "weights_kept": bool(keep), "total_sec": round(time.time() - t0, 2),
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    }
    append_jsonl(RUNS_JSONL, rec)
    del model
    return rec


def fmt_eta(sec):
    return str(timedelta(seconds=int(sec)))


# ---------------------------------------------------------------- main loop
queue = TODO[:LIMIT_RUNS] if LIMIT_RUNS else TODO
n_ok = n_fail = 0
recent = []                                        # durations of the last 50 runs, for the ETA
session_t0 = time.time()
config_runs = defaultdict(list)                    # completed folds per config
for rid, r in DONE.items():
    config_runs[r["experiment_id"]].append(r)
configs_done_before = sum(1 for c in CONFIGS if len(config_runs[experiment_id(c)]) == c["n_folds"])
configs_done = 0

print(f"Start: {len(queue)} runs queued, {len(DONE)} already done | {datetime.now().strftime('%H:%M:%S')}")
bar = tqdm(total=len(PLAN), initial=len(DONE), unit="run", dynamic_ncols=True)

for c, fold in queue:
    eid, rid = experiment_id(c), run_id_of(c, fold)
    t0 = time.time()
    try:
        rec = do_run(c, fold)
        n_ok += 1
        DONE[rid] = rec
        config_runs[eid].append(rec)
    except Exception as e:                          # a single failure does not stop the loop
        n_fail += 1
        append_jsonl(ERRORS_JSONL, {"run_id": rid, "experiment_id": eid, "fold": fold, "error": repr(e),
                                    "traceback": traceback.format_exc()[-2000:],
                                    "at": datetime.now().isoformat(timespec="seconds")})
        print(f"  ❌ {rid} failed: {repr(e)[:120]} (logged, continuing)")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        bar.update(1)
        continue

    dt = time.time() - t0
    recent = (recent + [dt])[-50:]
    remaining = len(queue) - n_ok - n_fail
    eta = np.mean(recent) * remaining
    bar.update(1)
    bar.set_postfix_str(f"{dt:.1f}s/run  ETA {fmt_eta(eta)}  failed {n_fail}")

    # config finished → print one line
    if len(config_runs[eid]) == c["n_folds"]:
        configs_done += 1
        rs = config_runs[eid]
        if c["shift"] == "small":
            r = rs[0]
            print(f"  ✅ [{configs_done_before + configs_done:>3}/272] {c['start']}→{c['target']} n={c['data_level']:<2} {c['layer_scheme']:<15} "
                  f"| target AUC {r['target_auc']:.4f} (frozen {r['frozen_target_auc']:.4f}, gain {r['target_auc_gain']:+.4f}) "
                  f"| source AUC {r['source_auc']:.4f} (forget {r['source_auc_forget']:+.4f}) | {r['total_sec']:.1f}s")
        else:
            aucs = [r["target_auc"] for r in rs]
            srcs = [r["source_auc"] for r in rs]
            gains = [r["target_auc_gain"] for r in rs]
            print(f"  ✅ [{configs_done_before + configs_done:>3}/272] {c['start']}→fan n={c['data_level']:<2} {c['layer_scheme']:<15} "
                  f"| fold AUC {np.mean(aucs):.4f}±{np.std(aucs):.4f} (gain {np.mean(gains):+.4f}) "
                  f"| source AUC {np.mean(srcs):.4f} (forget {BASE_SRC_AUC[c['start']] - np.mean(srcs):+.4f}) "
                  f"| {sum(r['total_sec'] for r in rs):.0f}s")
        if configs_done % 10 == 0:
            print(f"     ── session: {n_ok} runs / {fmt_eta(time.time() - session_t0)}, "
                  f"avg {np.mean(recent):.1f}s/run, est. remaining {fmt_eta(eta)} ──")

bar.close()
print(f"\nThis session: ok {n_ok} | failed {n_fail} | elapsed {fmt_eta(time.time() - session_t0)}")
print(f"Total completed {len(DONE)}/{len(PLAN)} runs" + ("  🎉 all done" if len(DONE) == len(PLAN) else
      f"  ← {len(PLAN) - len(DONE)} runs remaining; " + ("LIMIT_RUNS is active, set it back to None and run again" if LIMIT_RUNS else "re-running this notebook resumes automatically")))
if n_fail:
    print(f"⚠️ {n_fail} runs failed, see {ERRORS_JSONL}; re-running this notebook retries them automatically")

## 10 · Summary

Derived from `stage2_runs.jsonl` (**this cell can be re-run on its own at any time**, even if the main loop has only partly finished):

- `stage2_runs.csv` — 992 rows, one per training run
- `stage2_configs.csv` — 272 rows, one per config. The large-shift pooled metrics are recomputed from the 10 folds' target scores in `clip_scores/`

Plus two condensed tables (layer scheme × data level), and a list of failures.

**Reading note**: the best-F1 floor is 0.800, so AUC is primary; positive `gain` = better than frozen, positive `forget` = worse than Stage 0.

In [ ]:
DONE = load_done()
runs_df = pd.DataFrame(list(DONE.values()))
if runs_df.empty:
    raise RuntimeError("stage2_runs.jsonl is empty, no results to summarize yet")
runs_df = runs_df.drop(columns=["loss_curve"], errors="ignore").sort_values("run_id")
runs_df.to_csv(os.path.join(STAGE2_LOGS, "stage2_runs.csv"), index=False)

# ---- Config-level summary ----
cfg_rows, incomplete = [], []
for c in CONFIGS:
    eid = experiment_id(c)
    rs = [r for r in DONE.values() if r["experiment_id"] == eid]
    if len(rs) != c["n_folds"]:
        incomplete.append((eid, len(rs), c["n_folds"]))
        continue
    base = {"experiment_id": eid, "shift": c["shift"], "start": c["start"], "target": c["target"],
            "data_level": c["data_level"], "layer_scheme": c["layer_scheme"], "n_runs": len(rs),
            "base_source_auc": BASE_SRC_AUC[c["source_eval"]],
            "drift_encoder_mean": np.mean([r["drift_encoder_mean"] for r in rs]),
            "drift_decoder_mean": np.mean([r["drift_decoder_mean"] for r in rs]),
            "n_trainable_params": rs[0]["n_trainable_params"], "loss_last_mean": np.mean([r["loss_last"] for r in rs])}
    if c["shift"] == "small":
        r = rs[0]
        cfg_rows.append({**base,
            "target_auc": r["target_auc"], "target_best_f1": r["target_best_f1"],
            "frozen_target_auc": r["frozen_target_auc"], "target_auc_gain": r["target_auc_gain"], "target_f1_gain": r["target_f1_gain"],
            "source_auc": r["source_auc"], "source_best_f1": r["source_best_f1"],
            "source_auc_forget": r["source_auc_forget"], "source_f1_forget": r["source_f1_forget"]})
    else:
        # pooled: concatenate the 10 folds' target scores into 180 clips
        sc = pd.read_csv(os.path.join(STAGE2_SCORES, f"{eid}.csv"))
        sc = sc[sc["domain"] == "target"].drop_duplicates(subset=["run_id", "file"], keep="last")
        assert len(sc) == TGT["n_clips_total"] and sc["fold"].nunique() == LORO["n_folds"], f"{eid} scores for pooling are incomplete: {len(sc)} rows"
        pooled_auc = float(metrics.roc_auc_score(sc["label"], sc["score"]))
        pooled_f1 = best_f1_scan(sc["label"].values, sc["score"].values)["f1"]
        fz = FROZEN_LARGE[c["start"]]
        fa = [r["target_auc"] for r in rs]; ff = [r["target_best_f1"] for r in rs]
        sa = [r["source_auc"] for r in rs]; sf = [r["source_best_f1"] for r in rs]
        cfg_rows.append({**base,
            "pooled_auc": pooled_auc, "pooled_best_f1": pooled_f1,
            "frozen_pooled_auc": fz["pooled_auc"], "pooled_auc_gain": pooled_auc - fz["pooled_auc"],
            "pooled_f1_gain": pooled_f1 - fz["pooled_best_f1"],
            "fold_auc_mean": np.mean(fa), "fold_auc_std": np.std(fa), "fold_f1_mean": np.mean(ff), "fold_f1_std": np.std(ff),
            "fold_auc_gain_mean": np.mean([r["target_auc_gain"] for r in rs]),
            "source_auc": np.mean(sa), "source_auc_std": np.std(sa), "source_best_f1": np.mean(sf),
            "source_auc_forget": BASE_SRC_AUC[c["start"]] - np.mean(sa), "source_f1_forget": BASE_SRC_F1[c["start"]] - np.mean(sf)})

cfg_df = pd.DataFrame(cfg_rows)
cfg_df.to_csv(os.path.join(STAGE2_LOGS, "stage2_configs.csv"), index=False)
print(f"Configs complete {len(cfg_rows)}/272 (incomplete {len(incomplete)}), training runs {len(DONE)}/992")
if incomplete:
    print("  Incomplete configs (first 10): " + "; ".join(f"{e} {a}/{b}" for e, a, b in incomplete[:10]))

# ---- Condensed tables: layer scheme × data level ----
def pivot(df, val, title):
    if df.empty:
        return
    t = df.pivot_table(index="layer_scheme", columns="data_level", values=val, aggfunc="mean")
    t = t.reindex(list(SCHEMES)).reindex(columns=LEVELS)
    print(f"\n{title}")
    print("-" * 60)
    print(f"{'layer scheme':<18}" + "".join(f"{'n=' + str(n):>10}" for n in LEVELS))
    for s, row in t.iterrows():
        print(f"{s:<18}" + "".join(f"{v:>+10.4f}" if pd.notna(v) else f"{'—':>10}" for v in row))

small_c = cfg_df[cfg_df["shift"] == "small"] if not cfg_df.empty else cfg_df
large_c = cfg_df[cfg_df["shift"] == "large"] if not cfg_df.empty else cfg_df

print("\n" + "=" * 60 + "\nSmall shift (mean over 12 start→target pairs; mean frozen-baseline AUC "
      + f"{np.mean([v['auc'] for v in FROZEN_SMALL.values()]):.4f})\n" + "=" * 60)
pivot(small_c, "target_auc_gain", "Target AUC gain (vs. Stage 1 frozen)")
pivot(small_c, "source_auc_forget", "Source AUC forgetting (vs. Stage 0)")

print("\n" + "=" * 60 + "\nLarge shift (mean over 5 starts; mean frozen-baseline pooled AUC "
      + f"{np.mean([v['pooled_auc'] for v in FROZEN_LARGE.values()]):.4f})\n" + "=" * 60)
pivot(large_c, "pooled_auc_gain", "Target pooled AUC gain (vs. Stage 1 frozen)")
pivot(large_c, "source_auc_forget", "Source AUC forgetting (vs. Stage 0, 10-fold mean)")
pivot(large_c, "fold_auc_std", "Per-fold AUC std (stability)")

if not large_c.empty:
    print("\nLarge shift · pooled AUC per start at n=40/full (merged vs single-ID)")
    sub = large_c[(large_c["data_level"] == max(LEVELS)) & (large_c["layer_scheme"] == "full")]
    for r in sub.itertuples():
        print(f"  {r.start:<8} frozen {r.frozen_pooled_auc:.4f} → fine-tuned {r.pooled_auc:.4f} ({r.pooled_auc_gain:+.4f}) | forget {r.source_auc_forget:+.4f}")

# ---- Failure list ----
if os.path.exists(ERRORS_JSONL):
    errs = [json.loads(l) for l in open(ERRORS_JSONL) if l.strip()]
    still = [e for e in errs if e["run_id"] not in DONE]
    print(f"\n{len(errs)} failure records, {len(still)} still unresolved" + (":" if still else ""))
    for e in still[:20]:
        print(f"  {e['run_id']}: {e['error'][:100]}")

print(f"\nOutputs ({STAGE2_LOGS}): stage2_runs.jsonl (source of truth), stage2_runs.csv, stage2_configs.csv, "
      f"clip_scores/ ({len(os.listdir(STAGE2_SCORES))} files), stage2_errors.jsonl (if any)")
print(f"Weights ({STAGE2_MODELS}): {len(glob.glob(os.path.join(STAGE2_MODELS, '*.pth')))} files")
cfg_df.head(10)